# Session 2

In [1]:
import QuantLib as ql
import pandas as pd
import numpy as np


def build_demo_sofr_curve(
    evaluation_date: ql.Date | None = None,
    curve_type: str = "log_cubic_discount",
):
    """
    使用虚构的 SOFR OIS 市场报价构建一条 demo SOFR 曲线。

    Parameters
    ----------
    evaluation_date
        QuantLib evaluation date。
    curve_type
        支持：
        - "log_cubic_discount"
        - "linear_zero"
        - "flat_forward"

    Returns
    -------
    sofr_curve
        QuantLib YieldTermStructure 实例。
    market_quotes
        保存 demo OIS 报价及对应 Quote 对象的 DataFrame。
        保留 SimpleQuote 引用后，可以动态更新市场报价。
    """

    if evaluation_date is None:
        evaluation_date = ql.Date.todaysDate()

    ql.Settings.instance().evaluationDate = evaluation_date

    # 曲线的时间计算基准。
    curve_day_count = ql.Actual360()

    # 用于 bootstrap 的 SOFR index。
    # 此时不需要提前绑定曲线，QuantLib 会在 bootstrap 过程中处理。
    bootstrap_sofr_index = ql.Sofr()

    # ---------------------------------------------------------
    # Demo SOFR OIS par rates
    #
    # 注意：
    # 1. 以下报价完全是示例数据，不代表真实市场；
    # 2. 利率必须使用小数，例如 5.30% 写成 0.0530。
    # ---------------------------------------------------------
    demo_ois_quotes = [
        ("1W",  0.0530),
        ("1M",  0.0528),
        ("3M",  0.0520),
        ("6M",  0.0505),
        ("1Y",  0.0480),
        ("2Y",  0.0440),
        ("3Y",  0.0420),
        ("5Y",  0.0405),
        ("7Y",  0.0400),
        ("10Y", 0.0400),
        ("15Y", 0.0408),
        ("20Y", 0.0415),
        ("30Y", 0.0420),
    ]

    rate_helpers = []
    quote_rows = []

    settlement_days = 2

    for tenor_string, rate in demo_ois_quotes:
        simple_quote = ql.SimpleQuote(rate)
        quote_handle = ql.QuoteHandle(simple_quote)
        tenor = ql.Period(tenor_string)

        helper = ql.OISRateHelper(
            settlement_days,
            tenor,
            quote_handle,
            bootstrap_sofr_index,
        )

        rate_helpers.append(helper)

        quote_rows.append(
            {
                "tenor": tenor_string,
                "rate": rate,
                "rate_percent": rate * 100.0,
                "simple_quote": simple_quote,
                "quote_handle": quote_handle,
                "helper": helper,
            }
        )

    # ---------------------------------------------------------
    # 选择曲线类型
    # ---------------------------------------------------------
    curve_builders = {
        "log_cubic_discount": ql.PiecewiseLogCubicDiscount,
        "linear_zero": ql.PiecewiseLinearZero,
        "flat_forward": ql.PiecewiseFlatForward,
    }

    try:
        curve_builder = curve_builders[curve_type]
    except KeyError as exc:
        supported = ", ".join(curve_builders)
        raise ValueError(
            f"不支持的 curve_type={curve_type!r}。支持：{supported}"
        ) from exc

    sofr_curve = curve_builder(
        evaluation_date,
        rate_helpers,
        curve_day_count,
    )

    # 允许查询最后一个 bootstrap pillar 之后的期限。
    sofr_curve.enableExtrapolation()

    # 主动触发 bootstrap，便于尽早发现报价或 helper 问题。
    sofr_curve.discount(evaluation_date)

    market_quotes = pd.DataFrame(quote_rows)

    return sofr_curve, market_quotes


# =============================================================
# 构建 SOFR 曲线
# =============================================================
evaluation_date = ql.Date(23, ql.July, 2026)

sofr_curve, sofr_market_quotes = build_demo_sofr_curve(
    evaluation_date=evaluation_date,
    curve_type="log_cubic_discount",
)

In [ ]:
sofr_curve_handle = ql.YieldTermStructureHandle(sofr_curve)
sofr_index = ql.Sofr(sofr_curve_handle)
discount_engine = ql.DiscountingSwapEngine(sofr_curve_handle)

In [7]:
def build_sofr_ois(
    curve_handle: ql.YieldTermStructureHandle,
    effective_date: ql.Date,
    tenor: ql.Period,
    fixed_rate: float,
    notional: float = 100_000_000.0,
    swap_type: int = ql.Swap.Payer,
    payment_frequency: int = ql.Annual,
    fixed_day_count: ql.DayCounter = ql.Actual360(),
    payment_lag: int = 0,
    telescopic_value_dates: bool = True,
) -> ql.OvernightIndexedSwap:
    """
    Build and price a SOFR OIS.
    """

    sofr = ql.Sofr(curve_handle)
    calendar = sofr.fixingCalendar()

    swap = ql.MakeOIS(
        swapTenor=tenor,
        overnightIndex=sofr,
        fixedRate=fixed_rate,

        effectiveDate=effective_date,
        paymentFrequency=payment_frequency,
        swapType=swap_type,
        nominal=notional,
        fixedLegDayCount=fixed_day_count,

        paymentLag=payment_lag,
        paymentCalendar=calendar,

        # 修正：不是 paymentAdjustment
        paymentAdjustmentConvention=ql.Following,

        calendar=calendar,
        convention=ql.ModifiedFollowing,
        terminationDateConvention=ql.ModifiedFollowing,
        endOfMonth=False,

        telescopicValueDates=telescopic_value_dates,
        averagingMethod=ql.RateAveraging.Compound,
        lookbackDays=0,
        lockoutDays=0,
        applyObservationShift=False,
    )

    swap.setPricingEngine(
        ql.DiscountingSwapEngine(curve_handle)
    )

    return swap

In [8]:
settlement_days = 2

spot_date = sofr_index.fixingCalendar().advance(
    evaluation_date,
    settlement_days,
    ql.Days,
)

print("Evaluation date:", evaluation_date)
print("Spot date:", spot_date)
print(
    "Spot is business day:",
    sofr_index.fixingCalendar().isBusinessDay(spot_date),
)

Evaluation date: July 23rd, 2026
Spot date: July 27th, 2026
Spot is business day: True


In [9]:
def build_par_sofr_ois(
    curve_handle,
    effective_date,
    tenor,
    notional=100_000_000.0,
    swap_type=ql.Swap.Payer,
    **kwargs,
):
    probe = build_sofr_ois(
        curve_handle=curve_handle,
        effective_date=effective_date,
        tenor=tenor,
        fixed_rate=0.0,
        notional=notional,
        swap_type=swap_type,
        **kwargs,
    )

    fair_rate = probe.fairRate()

    par_swap = build_sofr_ois(
        curve_handle=curve_handle,
        effective_date=effective_date,
        tenor=tenor,
        fixed_rate=fair_rate,
        notional=notional,
        swap_type=swap_type,
        **kwargs,
    )

    return par_swap, fair_rate

In [10]:
tenors = {
    "2Y": ql.Period(2, ql.Years),
    "5Y": ql.Period(5, ql.Years),
    "10Y": ql.Period(10, ql.Years),
}

par_swaps = {}
rows = []

for name, tenor in tenors.items():
    swap, fair_rate = build_par_sofr_ois(
        curve_handle=sofr_curve_handle,
        effective_date=spot_date,
        tenor=tenor,
        notional=100_000_000.0,
        swap_type=ql.Swap.Payer,
        payment_frequency=ql.Annual,
        payment_lag=0,
    )

    par_swaps[name] = swap

    rows.append(
        {
            "tenor": name,
            "effective_date": spot_date.ISO(),
            "maturity_date": swap.maturityDate().ISO(),
            "fair_rate": fair_rate,
            "npv": swap.NPV(),
            "fixed_leg_npv": swap.fixedLegNPV(),
            "overnight_leg_npv": swap.overnightLegNPV(),
            "fixed_leg_bps": swap.fixedLegBPS(),
            "overnight_leg_bps": swap.overnightLegBPS(),
        }
    )

summary = pd.DataFrame(rows)
summary

,tenor,effective_date,maturity_date,fair_rate,npv,fixed_leg_npv,overnight_leg_npv,fixed_leg_bps,overnight_leg_bps
0,2Y,2026-07-27,2028-07-27,0.0440,0.000000e+00,-8.348802e+06,8.348802e+06,-18974.549861,18974.549861
1,5Y,2026-07-27,2031-07-28,0.0405,3.725290e-09,-1.816263e+07,1.816263e+07,-44845.991139,44845.991139
2,10Y,2026-07-27,2036-07-28,0.0400,-1.862645e-08,-3.273562e+07,3.273562e+07,-81839.045817,81839.045817


In [13]:
notional = 100_000_000.0

swap_5y, _ = build_par_sofr_ois(
    curve_handle=sofr_curve_handle,
    effective_date=spot_date,
    tenor=tenors["5Y"],
    notional=notional,
    swap_type=ql.Swap.Payer,
    payment_frequency=ql.Annual,
    payment_lag=0,
)

annuity_5y = (
    abs(swap_5y.fixedLegBPS())
    / (notional * 1e-4)
)

print("5Y swap annuity:", annuity_5y)

fair_rate_5y = swap_5y.fairRate()

fixed_pv_from_formula = (
    notional
    * fair_rate_5y
    * annuity_5y
)

print(
    "Formula fixed PV:",
    fixed_pv_from_formula,
)

print(
    "QuantLib fixed PV:",
    abs(swap_5y.fixedLegNPV()),
)

5Y swap annuity: 4.484599113879283
Formula fixed PV: 18162626.411168087
QuantLib fixed PV: 18162626.41116808


In [ ]:
fair_5y = swap_5y.fairRate()
contract_rate = fair_5y - 25e-4

payer_5y = build_sofr_ois(
    curve_handle=sofr_curve_handle,
    effective_date=spot_date,
    tenor=ql.Period(5, ql.Years),
    fixed_rate=contract_rate,
    notional=100_000_000.0,
    swap_type=ql.Swap.Payer,
)

print("Contract rate:", contract_rate)
print("Current fair rate:", payer_5y.fairRate())
print("Payer NPV:", payer_5y.NPV())

Contract rate: 0.0379999999999041
Current fair rate: 0.040499999999904106
Payer NPV: 1121149.7784698233


In [15]:
annuity = (
    abs(payer_5y.fixedLegBPS())
    / (100_000_000.0 * 1e-4)
)

formula_npv = (
    100_000_000.0
    * annuity
    * (
        payer_5y.fairRate()
        - contract_rate
    )
)

print("QuantLib NPV:", payer_5y.NPV())
print("Formula NPV:", formula_npv)
print(
    "Difference:",
    payer_5y.NPV() - formula_npv,
)

QuantLib NPV: 1121149.7784698233
Formula NPV: 1121149.778469825
Difference: -1.6298145055770874e-09


In [16]:
def fixed_leg_table(
    swap: ql.OvernightIndexedSwap,
    curve_handle: ql.YieldTermStructureHandle,
    swap_type: int,
) -> pd.DataFrame:
    curve = curve_handle.currentLink()

    fixed_sign = (
        -1.0
        if swap_type == ql.Swap.Payer
        else 1.0
    )

    rows = []

    for cf in swap.fixedLeg():
        coupon = ql.as_fixed_rate_coupon(cf)

        payment_date = coupon.date()
        discount_factor = curve.discount(payment_date)
        amount = coupon.amount()

        rows.append(
            {
                "accrual_start": coupon.accrualStartDate().ISO(),
                "accrual_end": coupon.accrualEndDate().ISO(),
                "payment_date": payment_date.ISO(),
                "accrual_fraction": coupon.accrualPeriod(),
                "rate": coupon.rate(),
                "amount": amount,
                "discount_factor": discount_factor,
                "signed_pv": (
                    fixed_sign
                    * amount
                    * discount_factor
                ),
            }
        )

    return pd.DataFrame(rows)

In [17]:
fixed_cf = fixed_leg_table(
    payer_5y,
    sofr_curve_handle,
    ql.Swap.Payer,
)

fixed_cf

,accrual_start,accrual_end,payment_date,accrual_fraction,rate,amount,discount_factor,signed_pv
0,2026-07-27,2027-07-27,2027-07-27,1.013889,0.038,3.852778e+06,0.953030,-3.671813e+06
1,2027-07-27,2028-07-27,2028-07-27,1.016667,0.038,3.863333e+06,0.915923,-3.538516e+06
2,2028-07-27,2029-07-27,2029-07-27,1.013889,0.038,3.852778e+06,0.882153,-3.398739e+06
3,2029-07-27,2030-07-29,2030-07-29,1.019444,0.038,3.873889e+06,0.849353,-3.290298e+06
4,2030-07-29,2031-07-28,2031-07-28,1.011111,0.038,3.842222e+06,0.817785,-3.142110e+06


In [18]:
print(
    "Manual fixed-leg PV:",
    fixed_cf["signed_pv"].sum(),
)

print(
    "QuantLib fixed-leg NPV:",
    payer_5y.fixedLegNPV(),
)

Manual fixed-leg PV: -17041476.63269826
QuantLib fixed-leg NPV: -17041476.63269826


In [19]:
def overnight_leg_table(
    swap: ql.OvernightIndexedSwap,
    curve_handle: ql.YieldTermStructureHandle,
    swap_type: int,
) -> pd.DataFrame:
    curve = curve_handle.currentLink()

    overnight_sign = (
        1.0
        if swap_type == ql.Swap.Payer
        else -1.0
    )

    rows = []

    for cf in swap.overnightLeg():
        coupon = ql.as_floating_rate_coupon(cf)

        payment_date = coupon.date()
        discount_factor = curve.discount(payment_date)
        amount = coupon.amount()

        rows.append(
            {
                "accrual_start": coupon.accrualStartDate().ISO(),
                "accrual_end": coupon.accrualEndDate().ISO(),
                "payment_date": payment_date.ISO(),
                "accrual_fraction": coupon.accrualPeriod(),
                "compound_rate": coupon.rate(),
                "amount": amount,
                "discount_factor": discount_factor,
                "signed_pv": (
                    overnight_sign
                    * amount
                    * discount_factor
                ),
            }
        )

    return pd.DataFrame(rows)

In [20]:
overnight_cf = overnight_leg_table(
    payer_5y,
    sofr_curve_handle,
    ql.Swap.Payer,
)

overnight_cf

,accrual_start,accrual_end,payment_date,accrual_fraction,compound_rate,amount,discount_factor,signed_pv
0,2026-07-27,2027-07-27,2027-07-27,1.013889,0.048000,4.866667e+06,0.953030,4.638080e+06
1,2027-07-27,2028-07-27,2028-07-27,1.016667,0.039849,4.051347e+06,0.915923,3.710722e+06
2,2028-07-27,2029-07-27,2029-07-27,1.013889,0.037757,3.828146e+06,0.882153,3.377010e+06
3,2029-07-27,2030-07-29,2030-07-29,1.019444,0.037881,3.861767e+06,0.849353,3.280003e+06
4,2030-07-29,2031-07-28,2031-07-28,1.011111,0.038178,3.860199e+06,0.817785,3.156812e+06


In [21]:
print(
    "Manual overnight-leg PV:",
    overnight_cf["signed_pv"].sum(),
)

print(
    "QuantLib overnight-leg NPV:",
    payer_5y.overnightLegNPV(),
)

Manual overnight-leg PV: 18162626.411168084
QuantLib overnight-leg NPV: 18162626.411168084


# Session 3

In [22]:
import QuantLib as ql

evaluation_date = ql.Settings.instance().evaluationDate
calendar = sofr_index.fixingCalendar()

spot_date = calendar.advance(
    evaluation_date,
    2,
    ql.Days,
)

forward_start_date = calendar.advance(
    spot_date,
    ql.Period(1, ql.Years),
    ql.ModifiedFollowing,
    False,
)

forward_end_date = calendar.advance(
    forward_start_date,
    ql.Period(5, ql.Years),
    ql.ModifiedFollowing,
    False,
)

print("Evaluation:", evaluation_date.ISO())
print("Spot:", spot_date.ISO())
print("1Y5Y start:", forward_start_date.ISO())
print("1Y5Y end:", forward_end_date.ISO())

Evaluation: 2026-07-23
Spot: 2026-07-27
1Y5Y start: 2027-07-27
1Y5Y end: 2032-07-27


In [23]:
notional = 100_000_000.0

sofr_index = ql.Sofr(sofr_curve_handle)

discount_engine = ql.DiscountingSwapEngine(
    sofr_curve_handle
)

In [24]:
probe_1y5y = ql.MakeOIS(
    ql.Period(5, ql.Years),
    sofr_index,
    0.0,
    effectiveDate=forward_start_date,
    terminationDate=forward_end_date,
    swapType=ql.OvernightIndexedSwap.Payer,
    nominal=notional,
    paymentFrequency=ql.Annual,
    fixedLegDayCount=ql.Actual360(),
    paymentLag=0,
    paymentCalendar=calendar,
    telescopicValueDates=False,
)

probe_1y5y.setPricingEngine(discount_engine)

fair_rate_1y5y = probe_1y5y.fairRate()

par_1y5y = ql.MakeOIS(
    ql.Period(5, ql.Years),
    sofr_index,
    fair_rate_1y5y,
    effectiveDate=forward_start_date,
    terminationDate=forward_end_date,
    swapType=ql.OvernightIndexedSwap.Payer,
    nominal=notional,
    paymentFrequency=ql.Annual,
    fixedLegDayCount=ql.Actual360(),
    paymentLag=0,
    paymentCalendar=calendar,
    telescopicValueDates=False,
)

par_1y5y.setPricingEngine(discount_engine)

print("Fair rate:", fair_rate_1y5y)
print("NPV:", par_1y5y.NPV())
print("Start:", par_1y5y.startDate().ISO())
print("Maturity:", par_1y5y.maturityDate().ISO())
print("Fixed-leg BPS:", par_1y5y.fixedLegBPS())

Fair rate: 0.03843139009426971
NPV: -1.1175870895385742e-08
Start: 2027-07-27
Maturity: 2032-07-27
Fixed-leg BPS: -43164.09528558427


In [25]:
spot_5y = build_par_sofr_ois(
    curve_handle=sofr_curve_handle,
    effective_date=spot_date,
    tenor=ql.Period(5, ql.Years),
)[1]

spot_6y = build_par_sofr_ois(
    curve_handle=sofr_curve_handle,
    effective_date=spot_date,
    tenor=ql.Period(6, ql.Years),
)[1]

print("5Y spot:", spot_5y)
print("6Y spot:", spot_6y)
print("1Y5Y forward:", fair_rate_1y5y)

5Y spot: 0.04049999999990409
6Y spot: 0.040181606941371566
1Y5Y forward: 0.03843139009426971
